In [1]:
import pandas as pd
from pprint import pprint
import json
import re
from typing import Dict, List, Any
import random

In [2]:
def read_jsonl_file(file_path:str)->List[Dict[str,Any]]:
    data_list=[]
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)  # Parse each line as JSON
            data_list.append(record)
    return data_list

In [3]:
def save_for_finetune(converted, filename="converted_dataset.jsonl"):
    """
    Save the converted dataset in JSONL format for LLM fine-tuning.

    Args:
        converted (list): List of schema dicts produced by preprocess_dataset.
        filename (str): Output file name.
    """
    with open(filename, "w", encoding="utf-8") as f:
        for sample in converted:
            # Each sample becomes one JSON line
            f.write(json.dumps(sample, ensure_ascii=False) + "\n")

In [4]:
def swap_goal_category_and_type(samples):
    """
    Swap goal_category and goal_type in each sample.

    Args:
        samples (list[dict]): List of sample dictionaries.

    Returns:
        list[dict]: Updated samples with swapped goal_category and goal_type.
    """
    swapped = []

    for sample in samples:
        new_sample = sample.copy()

        # Swap values
        goal_category = sample.get("goal_category", "")
        goal_type = sample.get("goal_type", "")

        new_sample["goal_category"] = goal_type
        new_sample["goal_type"] = goal_category

        swapped.append(new_sample)

    return swapped


# Example usage:
samples = [
    {
        "goal": "Intersectionality and Identity in 21st Century American Female Authors Study Plan",
        "goal_category": "one_time",
        "goal_type": "study",
        "time_horizon": 8,
        "description": "A detailed 8-day study plan...",
        "baseline": {
            "fixed_constraints": "",
            "physical_constraints": "",
            "non_negotiables": "",
        },
        "tasks": [
            {
                "index": 0,
                "task": "1",
                "description": "Begin by familiarizing yourself...",
                "is_repeatable": False,
                "repeat_frequency": 0,
                "gap_flag": False,
                "estimated_duration_minutes": {"min": 30, "max": 45},
            }
        ],
    }
]


In [5]:
workout_data = read_jsonl_file("final\dayWorkout_converted_dataset.jsonl")
planner_study_data = read_jsonl_file("final\plannerStudy_converted_dataset.jsonl")
gemini_data = read_jsonl_file("final\gemini_dataset.jsonl")
workouts_101_data = read_jsonl_file("final\workouts_101.jsonl")

In [6]:
random.shuffle(workout_data)
half_size = len(workout_data) // 2

complete_dataset = (
    planner_study_data + workout_data[:half_size] + gemini_data + workouts_101_data
)

In [7]:
complete_dataset=swap_goal_category_and_type(complete_dataset)

In [8]:
print(len(workout_data))
print(len(planner_study_data))
print(len(gemini_data))
print(len(workouts_101_data))
print(len(complete_dataset))

900
602
464
103
1619


In [9]:
save_for_finetune(complete_dataset,filename="final/complete_dataset.jsonl")

In [10]:
import json


def convert_sample(sample: dict) -> dict:
    # constraints = sample["baseline"]
    # constraint_parts = []
    # if constraints.get("physical_constraints"):
    #     constraint_parts.append(f"- Physical: {constraints['physical_constraints']}")
    # if constraints.get("fixed_constraints"):
    #     constraint_parts.append(f"- Fixed: {constraints['fixed_constraints']}")
    # if constraints.get("non_negotiables"):
    #     constraint_parts.append(f"- Non-negotiables: {constraints['non_negotiables']}")

    instruction = (
        f"Goal: {sample['goal']}\n"
        f"Category: {sample['goal_category']}\n"
        f"Type: {sample['goal_type']}\n"
        f"Time Horizon: {sample['time_horizon']} days\n"
    )
    # if constraint_parts:
    #     instruction += "Constraints:\n" + "\n".join(constraint_parts)

    # Output is the original structure as clean JSON string
    output_dict = {
        "goal": sample["goal"],
        "goal_category": sample["goal_category"],
        "goal_type": sample["goal_type"],
        "time_horizon": sample["time_horizon"],
        "tasks": sample["tasks"],
    }

    return {
        
        "instruction": instruction.strip(),
        "input": "",
        "output": "\n".join([
            "```json",
            json.dumps(output_dict, ensure_ascii=False),
            "```"]),
    }

In [11]:
test =convert_sample(complete_dataset[0])
print(test)

{'instruction': 'Goal: AP Environmental Science: Microplastic Pollution Study Plan\nCategory: study\nType: one_time\nTime Horizon: 7 days', 'input': '', 'output': '```json\n{"goal": "AP Environmental Science: Microplastic Pollution Study Plan", "goal_category": "study", "goal_type": "one_time", "time_horizon": 7, "tasks": [{"index": 0, "task": "Introduction and Overview", "description": "Start with an overview of microplastic pollution. Watch an introductory video to understand the basics. Use the Pomodoro technique to stay focused: 25 minutes of study followed by a 5-minute break. After the video, review flashcards to familiarize yourself with key terms.", "is_repeatable": false, "repeat_frequency": 0, "gap_flag": false, "estimated_duration_minutes": {"min": 30, "max": 45}}, {"index": 1, "task": "Sources and Types of Microplastics", "description": "Focus on the sources and types of microplastics. Watch a detailed video on how microplastics enter marine environments. Use spaced repetit